In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns

In [ ]:
data = pd.read_csv("./data/salary2.csv")
data.head()

분석목적: 학력, 교육연수, 혼인상태, 직업정보가 있는 연봉데이터셋을 이용해 연봉 예측하기
* 연봉이 5만달러 이상인지 아닌지
* age: 나이
* workclass: 고용형태
* education: 학력
* education-num: 교육연수
* marital-status: 혼인상태
* occupation: 직업
* relationship: 가족관계
* race: 인종
* sex: 성별
* capital-gain: 자산증가
* capital-loss: 자산감소
* hours-per-week: 주당 노동 시간
* native-country: 본국
* class: 연봉구분 - target(분석대상)

In [ ]:
data.info()

In [ ]:
data.describe()

# 결측값 탐색

In [ ]:
data.isna().sum()

In [ ]:
data.isna().sum() / len(data) * 100

In [ ]:
data['class'].value_counts()

In [ ]:
data[data['workclass'].isna()]

In [ ]:
data[data['occupation'].isna()]

In [ ]:
data[(data['workclass'].isna()) & (data['occupation'].isna()) & (data['class'] == " >50K")]

In [ ]:
265 / 11687 *100

In [ ]:
data['class'].unique()

In [ ]:
data.dropna()

In [ ]:
45222 / len(data) * 100

# 결측값의 비율이 약 7.5%이고 삭제 시 데이터 분포에 편향을 주지 않으므로 삭제

In [ ]:
data = data.dropna()
data = data.reset_index(drop=True)
data

In [ ]:
data['class'] = data['class'].apply(lambda x: 1 if x == ' >50K' else 0)
data

# 이상치 탐색

In [ ]:
data.describe()

In [ ]:
data['capital-gain'].value_counts()

In [ ]:
data['capital-gain'].plot(kind="hist")

# EDA

In [ ]:
data.info()

In [ ]:
obj_cols = data.select_dtypes(include='object')
num_cols = data.select_dtypes(exclude='object')

In [ ]:
ratio_result = data[['education', 'class']].groupby('education').mean().sort_values(by='class', ascending=False)
ratio_result[ratio_result['class'] > 0.40].index

In [ ]:
important_cols = []
for col in obj_cols:
    print("=" * 30, col, "=" * 30)
    print(col, f"nunique {obj_cols[col].nunique()}")
    print()
    print(obj_cols[col].value_counts())
    print()
    print(data[[col, 'class']].groupby(col).mean().sort_values(by='class', ascending=False))
    ratio_result = data[[col, 'class']].groupby(col).mean().sort_values(by='class', ascending=False)
    ratio_result = data[[col, 'class']].groupby(col).mean().sort_values(by='class', ascending=False)
    important_cols.append(ratio_result[ratio_result['class'] > 0.40].index)
    print()

# 수치형 변수들 EDA

In [ ]:
for col in num_cols.iloc[:, :-1]:
    print("=" * 30, col, "=" * 30)
    print(col, f"describe {num_cols[col].describe()}")
    print()
    num_cols[col].plot(kind='hist')
    plt.show()
    print()
    print(data[[col, 'class']].groupby(col).mean().sort_values(by='class', ascending=False))
    ratio_result = data[[col, 'class']].groupby(col).mean().sort_values(by='class', ascending=False)
    important_cols.append(ratio_result[ratio_result['class'] > 0.40].index)
    print()

In [ ]:
important_cols

In [ ]:
final_cols = []
for col in important_cols:
    if len(col) > 0:
        print(col.name, len(col) )
        final_cols.append(col.name)
#     print(len(col))
final_cols

In [ ]:
data['education'].value_counts()

In [ ]:
data['education-num'].value_counts()

In [ ]:
data_set1 = data[final_cols]
data_set1 = data_set1.drop("education", axis=1)
data_set1

In [ ]:
X = pd.get_dummies(data_set1, drop_first=True)
X

In [ ]:
y = data['class']
y

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, stratify=y, random_state=10)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
y_train.value_counts()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
dtc = DecisionTreeClassifier(random_state=10)
dtc.fit(X_train, y_train)
pred = dtc.predict(X_test)
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

# 모델 성능 튜닝(하이퍼파라미터 튜닝)

In [ ]:
for i in range(1, 11):
    dtc = DecisionTreeClassifier(max_depth=i, random_state=10)
    dtc.fit(X_train, y_train)
    pred = dtc.predict(X_test)
    print("="*30, f"max_depth: {i}", "="*30)
    print(accuracy_score(y_test, pred))
    print(classification_report(y_test, pred))
    print()

In [ ]:
dtc = DecisionTreeClassifier(max_depth=9, random_state=10)
dtc.fit(X_train, y_train)
pred = dtc.predict(X_test)
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

# 의사결정나무 시각화

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
plt.figure(figsize=(20,20))
plot_tree(dtc, feature_names=dtc.feature_names_in_, max_depth=3, fontsize=15, filled=True)
plt.show()